# UPI Fraud Rings on Colab: plan `cat`

CatBoost on the GPU. Needs a GPU runtime.

Before running: `upi_code_*.zip` and `upi_cache.zip` must be in Google Drive, folder `MyDrive/upi_colab/`. Then **Runtime > Run all**. Results are copied to `MyDrive/upi_colab/out/cat/` after every fit. **Disconnect the runtime when the last cell finishes**: compute units burn while it stays connected.

In [ ]:
PLAN = "cat"
import glob, os, shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/upi_colab"
W = "/content/upi"
shutil.rmtree(W, ignore_errors=True)
os.makedirs(W)
def unzip(pattern):
    files = sorted(glob.glob(f"{D}/{pattern}"), key=os.path.getmtime)
    assert files, f"{pattern} not found in {D}: upload it to Google Drive, folder upi_colab"
    print("using", files[-1])
    zipfile.ZipFile(files[-1]).extractall(W)
unzip("upi_cache*.zip")
unzip("upi_code*.zip")          # last, so the newest code wins
!nvidia-smi -L || echo "no GPU in this runtime"
!nproc; free -g | head -2

In [ ]:
# Same library versions as the laptop. If the pinned install fails, the session continues with Colab's own versions.
!pip -q install -r /content/upi/requirements.txt catboost==1.2.10 || echo "pinned install failed: using the versions Colab ships"

In [ ]:
!cd /content/upi && python -I colab/driver.py $PLAN --out $D/out/$PLAN

In [ ]:
!cat /content/upi/results_$PLAN.md; ls -la $D/out/$PLAN